# Apéndice · 4 de 4 — Jugar contra los motores más fuertes

Cierra el apéndice con lo mismo que la [09](../09_jugar_contra_el_motor.ipynb)
hace para los modelos de la memoria: un tablero acá adentro para **jugarles**.
No agrega ninguna métrica —todas están en la [notebook 3](lichess_3_elo.ipynb),
con su incertidumbre—; agrega poder contrastar a mano lo que esas tablas
afirman, sobre las posiciones que elija quien lea.

## Los tres motores

| motor | entrenado con | parámetros | Elo a 1 ply | Elo a 2 plies |
|---|---|---|---|---|
| **30M-chica** | 30 M de posiciones de Lichess | 2,9 M | 1511 ± 40 | **1762 ± 46** |
| **30M-grande** | los mismos 30 M, red de 16 bloques | 5,3 M | 1576 ± 40 | **1720 ± 44** |
| ResNet de la memoria | 2,3 M de posiciones del proyecto | 2,9 M | 1124 ± 56 | 1534 ± 40 |

(Elo contra la misma escalera de Stockfish que la 08 —1320, 1500 y 1700—, de la
sección 9 de la notebook 3 y de la sección 8 de la 08.)

Los dos primeros son los más fuertes del apéndice, y entre ellos no hay una
diferencia que las partidas puedan separar: la red grande baja el RMSE pero no
juega mejor. La ResNet de la memoria está como **referencia conocida**: es el
motor que mide la 08, así que lo que hagan los otros dos se juzga contra algo.

## Qué mirar

| | |
|---|---|
| **El panel comparativo** (sección 4) | Una posición y los tres motores al lado, con Stockfish arriba: lo que cada red opina de la posición, la jugada que elige y cuánto pierde esa jugada |
| **Las dos barras** del tablero | La evaluación de la red y la de Stockfish, en la misma escala y desde las blancas |
| **Modelo** | Se cambia de motor en mitad de la partida; con una posición fija, es la forma más directa de ver en qué se diferencian |
| **Profundidad** | 2 plies es la que jugó la escalera; 1 ply muestra la evaluación casi cruda |

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija: es la referencia del tablero (seccion 2), la
# lectura que se muestra al lado de la de la red.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
import torch

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')
print('Se usa:', dispositivo)

# Esta notebook anda igual en CPU: una jugada a profundidad 1 evalua unas 30
# posiciones y a profundidad 2 unas 1.000, que en CPU son decimas de segundo.
# El runtime con GPU esta bien igual, pero no hace falta pedirlo para jugar.

## 2. Los modelos

Los mejores checkpoints por validación de los dos brazos de 30 M de la
notebook 2 y la ResNet de la memoria. No hace falta bajar el dataset —el tablero
sólo necesita las redes—, así que la notebook arranca en menos de un minuto.

In [ ]:
import chess
import chess.engine

from chessdl import hf
from chessdl.config import load_config
from chessdl.engine.evaluator import Evaluator
from chessdl.engine.loader import load_from_hub

cfg = load_config()
token = hf.get_token()
repo_modelos = f'{cfg.output.hf_namespace}/{cfg.training.hf_models_repo}'

CORRIDAS = {
    '30M-chica':      'pruebas-lichess/30M-chica',
    '30M-grande':     'pruebas-lichess/30M-grande',
    'ResNet memoria': 'campana2-warmup',
}

evaluadores = {}
for nombre, corrida in CORRIDAS.items():
    modelo = load_from_hub(repo_modelos, corrida, token=token, device=dispositivo)
    evaluadores[nombre] = Evaluator(modelo, device=dispositivo)
    print(f'{nombre:<16}{evaluadores[nombre].describe()}')

# Stockfish como referencia, no como rival: el panel y el tablero lo consultan
# para mostrar su lectura al lado de la de cada red.
referencia = chess.engine.SimpleEngine.popen_uci('stockfish')
print('referencia      ', referencia.id.get('name', 'Stockfish'))

## 3. Las posiciones

Las cuatro de la 09 y una apertura, cada una elegida por algo que la notebook 3
midió:

- **Mate en 1** — `Ra8#`. Los tres lo juegan siempre, porque la posición que deja
  el mate es terminal y se resuelve por reglas. Lo que cambia es el **panel**: en
  la sección 6.2 de la notebook 3 los modelos del apéndice le dieron +1,000 a un
  mate en 1, y la ResNet de la memoria +0,92. El dump de Lichess tiene diez veces
  más mates que el dataset del proyecto, y se nota.
- **Dama colgada** — `Qxd5` toma un peón defendido por `c6`. A un ply la
  recaptura no está en el árbol, y la única defensa es que la red vea la dama en
  prise mirando el tablero quieto. Los modelos del apéndice leen esas tácticas
  mucho mejor que la ResNet de la memoria (la otra dama en prise de la 6.2: entre
  −674 y −1015 cp, contra −324). **Mirá dónde queda `Qxd5` en el ranking de cada
  uno a 1 ply.**
- **Final de torre** — rey y torre contra rey. Los modelos del apéndice lo
  evalúan como ganado sin dudar (−0,999 y −1,000 al mismo final con la torre
  negra, contra −0,85 de la memoria), pero ganarlo exige un plan de una docena de
  jugadas, y a 2 plies casi todas "mantienen la ventaja" igual de bien. Saber que
  está ganado no alcanza para ganarlo: es la posición donde más claro se ve el
  límite de una búsqueda corta.
- **Apertura italiana** — las aperturas son lo más denso del dump de Lichess, que
  reúne lo que la gente manda a analizar.
- **Inicial** — para jugar una partida entera con blancas. Para que abra el
  motor, elegí *Juego con negras* y apretá *Nueva*.

In [ ]:
POSICIONES = {
    'Mate en 1':         '6k1/5ppp/8/8/8/8/8/R5K1 w - - 0 1',
    'Dama colgada':      'rnbqkbnr/pp3ppp/2p5/3p4/3Q4/8/PPP1PPPP/RNB1KBNR w KQkq - 0 1',
    'Final de torre':    '8/8/8/5k2/8/8/4R3/4K3 w - - 0 1',
    'Apertura italiana': 'r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4',
    'Inicial':           'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1',
}

for nombre, fen in POSICIONES.items():
    assert chess.Board(fen).is_valid(), f'{nombre}: FEN invalido'
print(f'{len(POSICIONES)} posiciones, todas validas')

## 4. El panel comparativo

Una posición, los tres motores al lado y Stockfish a profundidad 12 arriba como
vara. Las columnas no dicen lo mismo, y esa diferencia es el tema del apéndice:

| columna | qué es | con qué se corresponde en la notebook 3 |
|---|---|---|
| **evaluación** | lo que la red opina de la posición, sin buscar, desde el que mueve | el **RMSE** |
| **jugada** | la que elige con búsqueda de 2 plies | — |
| **= SF** | si coincide con la mejor de Stockfish | el **acuerdo de jugada** |
| **pierde** | cuántos centipeones tira esa jugada, según Stockfish | el **ACPL** |

La pérdida sale de `move_quality`, la misma función con la que la notebook 3
calcula el ACPL sobre 400 posiciones. Acá es una sola posición por vez: sirve
para **ver** la diferencia, no para medirla.

In [ ]:
from chessdl.engine.match import move_quality
from chessdl.engine.search import search
from chessdl.normalize import value_to_cp

PROFUNDIDAD = 2                            # la de la escalera
LIMITE_SF = chess.engine.Limit(depth=12)   # la que etiqueto el dataset del proyecto


def comparar(nombre, fen):
    """Los tres motores sobre una posicion, con Stockfish arriba como vara."""
    tablero = chess.Board(fen)
    juegan = 'blancas' if tablero.turn == chess.WHITE else 'negras'
    print(f'\n{nombre}  --  juegan {juegan}')

    lectura = referencia.analyse(tablero, LIMITE_SF)
    mejor = lectura['pv'][0] if lectura.get('pv') else None
    marcador = lectura['score'].pov(tablero.turn)
    sf = f'mate en {marcador.mate()}' if marcador.is_mate() else f'{marcador.score():+d} cp'
    print(f"{'Stockfish d12':<16}{sf:>12}   mejor jugada: "
          f"{tablero.san(mejor) if mejor else '-'}")
    print('-' * 60)
    print(f"{'motor':<16}{'evaluacion':>12}{'jugada':>10}{'= SF':>7}{'pierde':>13}")

    for etiqueta, ev in evaluadores.items():
        # La evaluacion cruda de la red, desde el que mueve: lo que mide el RMSE.
        cruda = value_to_cp(float(ev.evaluate_board(tablero)))
        elegida = search(tablero, ev, depth=PROFUNDIDAD).move
        q = move_quality(ev, referencia, [fen], LIMITE_SF, depth=PROFUNDIDAD)
        print(f'{etiqueta:<16}{cruda:>+9.0f} cp{tablero.san(elegida):>10}'
              f'{"si" if q.agreements[0] else "no":>7}{q.losses_cp[0]:>10.0f} cp')


for nombre, fen in POSICIONES.items():
    comparar(nombre, fen)

## 5. El tablero

**Click en la pieza, click en el casillero**, como en la 09. Los destinos
legales aparecen marcados; para enrocar alcanza con clickear el rey y después la
torre, y al coronar el tablero pregunta qué pieza.

El selector de **Modelo** cambia de motor en mitad de la partida. Con una
posición cargada, pasar de uno a otro y mirar el ranking de jugadas es la forma
más directa de ver en qué se diferencian. La referencia de Stockfish corre a
profundidad 12, la que etiquetó el dataset del proyecto: para la ResNet de la
memoria la brecha entre las dos barras es su error; para los del apéndice
—entrenados contra las etiquetas de Lichess— es además la transferencia que la
notebook 3 mide.

In [ ]:
from chessdl.ui import PlayUI

tablero = PlayUI(evaluadores, depth=PROFUNDIDAD, positions=POSICIONES,
                 reference=referencia, reference_depth=12)
tablero

## 6. Cierre del apéndice

Las cuatro notebooks, y lo que dejó cada una:

| | |
|---|---|
| [1 · survey](lichess_1_survey.ipynb) | El dump de evaluaciones de Lichess tiene unos 270 M de posiciones; exigir profundidad 12 cuesta menos del 1 %, y el 16 % son mates |
| [2 · entrenamiento](lichess_2_entrenamiento.ipynb) | El RMSE baja un 7 % por cada duplicación de datos, sin aplanarse a 30 M; duplicar la profundidad de la red compra un 2 % |
| [3 · fuerza de juego](lichess_3_elo.ipynb) | Con 10 M ya se supera a la memoria en todas las medidas; con 30 M, unos 230 puntos de Elo a 2 plies y casi 400 a 1 ply, y empate con Stockfish a igual búsqueda |
| 4 · jugar | Verificarlo a mano |

La conclusión del bloque 4 era que **el techo lo ponía el dataset**. El apéndice
la confirma por el camino directo: con más posiciones el modelo evalúa mejor y
el motor juega mejor, y la palanca sigue siendo el dato más que la capacidad.

> **La salvedad que vale para todo el apéndice.** Los modelos del apéndice no
> sólo vieron más posiciones: vieron otras —etiquetadas por otro Stockfish, a
> otra profundidad, con diez veces más mates y más táctica—. Lo medido con la
> misma vara (el test del proyecto, las mismas posiciones, la misma escalera)
> es comparable; atribuir toda la diferencia al volumen, no.